In [1]:
# 1. INSTALL DEPENDENCIES (If not already present)
!pip install pandas numpy torch scikit-learn openpyxl

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from collections import defaultdict
import warnings
import os
from google.colab import files

warnings.filterwarnings("ignore")

In [2]:
# ─────────────────────────────────────────────
# 2. FILE UPLOAD HANDLING
# ─────────────────────────────────────────────
DATA_PATH = "Sample_100_VINs.xlsx"

if not os.path.exists(DATA_PATH):
    print(f"Please upload '{DATA_PATH}' to proceed.")
    uploaded = files.upload()
    # In case the uploaded filename is different, we rename or update DATA_PATH
    for fn in uploaded.keys():
        if fn.endswith('.xlsx'):
            DATA_PATH = fn
            print(f"User uploaded {fn} to be used as data source.")

Please upload 'Sample_100_VINs.xlsx' to proceed.


Saving dataset.xlsx to dataset.xlsx
User uploaded dataset.xlsx to be used as data source.


In [3]:
# ─────────────────────────────────────────────
# 3. CONFIG & HYPERPARAMETERS
# ─────────────────────────────────────────────
HYPERPARAMS = {
    "SEQ_LEN":              10,
    "TARGET_WINDOW":         5,
    "EMBED_DIM":            64,
    "HIDDEN_SIZE":         128,
    "NUM_LAYERS":            2,
    "DROPOUT":             0.3,
    "LR":                 1e-3,
    "EPOCHS":               30,
    "BATCH_SIZE":           64,
    "PATIENCE":              5,
    "FOCAL_GAMMA":           1,   # mild focal; 0 = plain BCE
    "K_EVAL":             [3, 5],
    "NO_FAULT_GAP_DAYS":    30,
    "N_TEST_VINS":          20,   # completely unseen at test time
    "N_VAL_VINS":           10,   # held-out for val (early stopping)
    "RANDOM_SEED":          42,
    "MARKOV_BLEND_WEIGHT": 0.4,   # weight of Markov in final prediction blend
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
# ─────────────────────────────────────────────
# 4. PREPROCESSING LOGIC
# ─────────────────────────────────────────────

def load_and_preprocess(path, hp):
    print("\n[1/5] Loading data...")
    df = pd.read_excel(path)
    df["min_date"] = pd.to_datetime(df["min_date"])
    df["max_date"]  = pd.to_datetime(df["max_date"])
    df = df.sort_values(["vin", "min_date"]).reset_index(drop=True)
    print(f"    {len(df):,} rows | {df['vin'].nunique()} VINs | {df['triplet'].nunique()} DTCs")

    global_freq = df["triplet"].value_counts()
    df["global_log_freq"] = np.log1p(df["triplet"].map(global_freq)).astype(np.float32)

    le = LabelEncoder()
    le.fit(list(df["triplet"].unique()) + ["[NO_FAULT]", "[PAD]"])
    NO_FAULT_IDX = int(le.transform(["[NO_FAULT]"])[0])
    PAD_IDX      = int(le.transform(["[PAD]"])[0])
    VOCAB_SIZE   = len(le.classes_)
    df["dtc_idx"] = le.transform(df["triplet"])

    no_fault_freq = 0.0

    grouped = []
    for vin, grp in df.groupby("vin", sort=False):
        for ts, bag in grp.groupby("min_date"):
            dtc_list = list(bag["dtc_idx"])
            grouped.append({
                "vin":      vin,
                "ts":       ts,
                "dtc_list": dtc_list,
                "primary":  dtc_list[0],
                "odo":      float(bag["first_odometer"].iloc[0]),
                "log_freq": float(bag["global_log_freq"].iloc[0]),
            })

    df_g = pd.DataFrame(grouped).sort_values(["vin", "ts"]).reset_index(drop=True)
    df_g["delta_t"]   = df_g.groupby("vin")["ts"].diff().dt.total_seconds().div(86400).fillna(0)
    df_g["delta_odo"] = df_g.groupby("vin")["odo"].diff().fillna(0)

    gap_thr  = hp["NO_FAULT_GAP_DAYS"]
    enriched = []
    for vin, grp in df_g.groupby("vin", sort=False):
        rows = grp.to_dict("records")
        for i, row in enumerate(rows):
            enriched.append(row)
            if i < len(rows) - 1:
                gap = (rows[i+1]["ts"] - row["ts"]).total_seconds() / 86400
                if gap > gap_thr:
                    enriched.append({
                        "vin": vin, "ts": row["ts"] + pd.Timedelta(days=gap_thr),
                        "dtc_list": [NO_FAULT_IDX], "primary": NO_FAULT_IDX,
                        "odo": row["odo"], "log_freq": no_fault_freq,
                        "delta_t": float(gap_thr), "delta_odo": 0.0,
                    })

    df_e = pd.DataFrame(enriched).sort_values(["vin", "ts"]).reset_index(drop=True)
    print(f"    After NO_FAULT injection: {len(df_e):,} timesteps | vocab: {VOCAB_SIZE}")
    return df_e, le, VOCAB_SIZE, NO_FAULT_IDX, PAD_IDX

def split_vins(df_e, hp):
    rng        = np.random.default_rng(hp["RANDOM_SEED"])
    vin_counts = df_e.groupby("vin").size().sort_values(ascending=False)
    all_vins   = vin_counts.index.tolist()
    n          = len(all_vins)

    n_test = hp["N_TEST_VINS"]
    n_val  = hp["N_VAL_VINS"]

    step_test = n // n_test
    test_vins = [all_vins[i * step_test] for i in range(n_test)]
    remaining = [v for v in all_vins if v not in set(test_vins)]

    step_val  = len(remaining) // n_val
    val_vins  = [remaining[i * step_val] for i in range(n_val)]
    train_vins = [v for v in remaining if v not in set(val_vins)]

    print(f"\n[2/5] VIN split: {len(train_vins)} train | {len(val_vins)} val | {len(test_vins)} test")
    return train_vins, val_vins, test_vins

def build_sequences(df_e, train_vins, val_vins, test_vins, hp, vocab_size):
    L, TW = hp["SEQ_LEN"], hp["TARGET_WINDOW"]
    def extract_windows(vin_list):
        samples = []
        for vin in vin_list:
            grp = df_e[df_e["vin"] == vin].sort_values("ts").reset_index(drop=True)
            n   = len(grp)
            if n < L + TW + 1: continue
            dtc_seq   = grp["primary"].values.astype(np.int64)
            dt_seq    = grp["delta_t"].values.astype(np.float32)
            dodo_seq  = grp["delta_odo"].values.astype(np.float32)
            freq_seq  = grp["log_freq"].values.astype(np.float32)
            dtc_lists = grp["dtc_list"].values
            for i in range(n - L - TW):
                target = np.zeros(vocab_size, dtype=np.float32)
                for j in range(i + L, i + L + TW):
                    for idx in dtc_lists[j]: target[idx] = 1.0
                samples.append((dtc_seq[i:i+L], dt_seq[i:i+L], dodo_seq[i:i+L], freq_seq[i:i+L], target))
        return samples

    splits = {"train": extract_windows(train_vins), "val": extract_windows(val_vins), "test": extract_windows(test_vins)}
    for k, v in splits.items(): print(f"    {k:5s}: {len(v):,} samples")
    return splits

In [5]:
# ─────────────────────────────────────────────
# 5. MODEL ARCHITECTURE
# ─────────────────────────────────────────────

class DTCDataset(Dataset):
    def __init__(self, samples): self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x_dtc, x_dt, x_dodo, x_freq, target = self.samples[idx]
        return (torch.tensor(x_dtc, dtype=torch.long), torch.tensor(x_dt, dtype=torch.float32),
                torch.tensor(x_dodo, dtype=torch.float32), torch.tensor(x_freq, dtype=torch.float32),
                torch.tensor(target, dtype=torch.float32))

class DTCPredictor(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.feat_proj = nn.Sequential(nn.Linear(3, embed_dim), nn.ReLU())
        self.lstm = nn.LSTM(input_size=embed_dim*2, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, dtc_seq, dt_seq, dodo_seq, freq_seq):
        emb = self.embedding(dtc_seq)
        feats = self.feat_proj(torch.stack([dt_seq, dodo_seq, freq_seq], dim=-1))
        x = torch.cat([emb, feats], dim=-1)
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

In [6]:
# ─────────────────────────────────────────────
# 6. MARKOV BLENDING
# ─────────────────────────────────────────────

def build_markov(df_e, train_vins, vocab_size, k=50):
    print("\n[3/5] Building Markov model...")
    train_df = df_e[df_e["vin"].isin(train_vins)]
    transitions = defaultdict(lambda: defaultdict(int))
    for vin, grp in train_df.groupby("vin", sort=False):
        seq = grp.sort_values("ts")["primary"].values
        for i in range(len(seq) - 1): transitions[seq[i]][seq[i+1]] += 1
    markov_probs, markov_top_k = {}, {}
    for src, counts in transitions.items():
        total = sum(counts.values())
        prob_vec = np.zeros(vocab_size, dtype=np.float32)
        for dst, cnt in counts.items(): prob_vec[dst] = cnt / total
        markov_probs[src] = prob_vec
        markov_top_k[src] = sorted(counts, key=counts.get, reverse=True)[:k]
    global_counts = np.zeros(vocab_size, dtype=np.float32)
    for src, counts in transitions.items():
        for dst, cnt in counts.items(): global_counts[dst] += cnt
    global_counts /= (global_counts.sum() + 1e-9)
    markov_probs["__global__"] = global_counts
    return markov_probs, markov_top_k

def eval_markov_standalone(splits, markov_top_k, k_list, label):
    results = {k: {"hits": 0, "total": 0} for k in k_list}
    for sample in splits[label]:
        x_dtc, target = sample[0], sample[-1]
        last = int(x_dtc[-1])
        preds = markov_top_k.get(last, [])
        true_set = set(np.where(target > 0)[0])
        for k in k_list:
            results[k]["total"] += 1
            if set(preds[:k]) & true_set: results[k]["hits"] += 1
    print(f"    Markov {label} Recall@5: {results[5]['hits']/results[5]['total']:.4f}")

In [7]:
# ─────────────────────────────────────────────
# 7. TRAINING & EVALUATION UTILS
# ─────────────────────────────────────────────

class FocalBCELoss(nn.Module):
    def __init__(self, gamma=1.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        if self.gamma > 0:
            prob = torch.sigmoid(logits)
            p_t = targets * prob + (1 - targets) * (1 - prob)
            bce = bce * (1 - p_t) ** self.gamma
        return bce.mean()

def recall_at_k_tensor(logits, targets, k):
    top_k = torch.topk(logits, k, dim=1).indices
    hits = 0
    for i in range(targets.shape[0]):
        if set(targets[i].nonzero(as_tuple=False).flatten().tolist()) & set(top_k[i].tolist()): hits += 1
    return hits / targets.shape[0]

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    for x_dtc, x_dt, x_dodo, x_freq, target in loader:
        x_dtc, x_dt, x_dodo, x_freq, target = x_dtc.to(device), x_dt.to(device), x_dodo.to(device), x_freq.to(device), target.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x_dtc, x_dt, x_dodo, x_freq), target)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion, device, k_list, markov_probs=None, blend_weight=0.0, vocab_size=None):
    model.eval()
    total_loss, all_logits, all_targets, all_last_dtc = 0.0, [], [], []
    for x_dtc, x_dt, x_dodo, x_freq, target in loader:
        logits = model(x_dtc.to(device), x_dt.to(device), x_dodo.to(device), x_freq.to(device))
        total_loss += criterion(logits, target.to(device)).item()
        all_logits.append(logits.cpu()); all_targets.append(target.cpu()); all_last_dtc.append(x_dtc[:, -1].cpu())
    all_logits, all_targets, all_last_dtc = torch.cat(all_logits), torch.cat(all_targets), torch.cat(all_last_dtc)
    lstm_probs = torch.sigmoid(all_logits)
    if blend_weight > 0 and markov_probs is not None:
        markov_mat = np.zeros((len(all_last_dtc), vocab_size), dtype=np.float32)
        global_fb = markov_probs["__global__"]
        for i, last in enumerate(all_last_dtc.tolist()): markov_mat[i] = markov_probs.get(last, global_fb)
        eval_scores = (1 - blend_weight) * lstm_probs + blend_weight * torch.tensor(markov_mat)
    else: eval_scores = lstm_probs
    metrics = {"loss": total_loss / len(loader)}
    for k in k_list: metrics[f"recall@{k}"] = recall_at_k_tensor(eval_scores, all_targets, k)
    return metrics

In [8]:
# ─────────────────────────────────────────────
# 8. EXECUTION
# ─────────────────────────────────────────────

if __name__ == "__main__":
    hp = HYPERPARAMS
    df_e, le, VOCAB_SIZE, NO_FAULT_IDX, PAD_IDX = load_and_preprocess(DATA_PATH, hp)
    train_vins, val_vins, test_vins = split_vins(df_e, hp)
    splits = build_sequences(df_e, train_vins, val_vins, test_vins, hp, VOCAB_SIZE)
    markov_probs, markov_top_k = build_markov(df_e, train_vins, VOCAB_SIZE)

    print("\n[4/5] Training LSTM model...")
    train_loader = DataLoader(DTCDataset(splits["train"]), batch_size=hp["BATCH_SIZE"], shuffle=True)
    val_loader = DataLoader(DTCDataset(splits["val"]), batch_size=hp["BATCH_SIZE"], shuffle=False)
    model = DTCPredictor(VOCAB_SIZE, hp["EMBED_DIM"], hp["HIDDEN_SIZE"], hp["NUM_LAYERS"], hp["DROPOUT"]).to(DEVICE)
    criterion = FocalBCELoss(gamma=hp["FOCAL_GAMMA"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=hp["LR"], weight_decay=1e-4)

    best_r5, best_state, patience_ctr = -1.0, None, 0
    for epoch in range(1, hp["EPOCHS"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_m = evaluate(model, val_loader, criterion, DEVICE, hp["K_EVAL"], markov_probs, hp["MARKOV_BLEND_WEIGHT"], VOCAB_SIZE)
        print(f"  Epoch {epoch:2d} | Train Loss: {train_loss:.4f} | Val R@5: {val_m['recall@5']:.4f}")
        if val_m['recall@5'] > best_r5:
            best_r5, best_state, patience_ctr = val_m['recall@5'], {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= hp["PATIENCE"]: break

    model.load_state_dict(best_state)
    print("\n[5/5] Final Testing...")
    test_loader = DataLoader(DTCDataset(splits["test"]), batch_size=hp["BATCH_SIZE"], shuffle=False)
    test_m = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), DEVICE, hp["K_EVAL"], markov_probs, hp["MARKOV_BLEND_WEIGHT"], VOCAB_SIZE)
    print(f"Final Test Recall@5 (Blended): {test_m['recall@5']:.4f}")

    torch.save({"model_state": model.state_dict(), "le": le, "markov": markov_probs}, "dtc_model_v3.pt")
    print("Model saved as dtc_model_v3.pt")


[1/5] Loading data...
    195,883 rows | 100 VINs | 1311 DTCs
    After NO_FAULT injection: 71,158 timesteps | vocab: 1313

[2/5] VIN split: 70 train | 10 val | 20 test
    train: 47,542 samples
    val  : 7,469 samples
    test : 14,647 samples

[3/5] Building Markov model...

[4/5] Training LSTM model...
  Epoch  1 | Train Loss: 0.0190 | Val R@5: 0.7688
  Epoch  2 | Train Loss: 0.0138 | Val R@5: 0.8012
  Epoch  3 | Train Loss: 0.0124 | Val R@5: 0.8423
  Epoch  4 | Train Loss: 0.0113 | Val R@5: 0.8605
  Epoch  5 | Train Loss: 0.0107 | Val R@5: 0.8596
  Epoch  6 | Train Loss: 0.0104 | Val R@5: 0.8605
  Epoch  7 | Train Loss: 0.0100 | Val R@5: 0.8494
  Epoch  8 | Train Loss: 0.0096 | Val R@5: 0.8593
  Epoch  9 | Train Loss: 0.0093 | Val R@5: 0.8711
  Epoch 10 | Train Loss: 0.0090 | Val R@5: 0.8602
  Epoch 11 | Train Loss: 0.0087 | Val R@5: 0.8464
  Epoch 12 | Train Loss: 0.0085 | Val R@5: 0.8487
  Epoch 13 | Train Loss: 0.0083 | Val R@5: 0.8388
  Epoch 14 | Train Loss: 0.0081 | Val R@5